In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

df = pd.read_parquet('../data/processed/protein_features_hol.parquet')

CATEGORICALS = ["family", "city", "state", "store_type", "store_nbr", "item_nbr", "class", "cluster"]
for c in CATEGORICALS:
    df[c] = df[c].astype("category")

test = df[df["date"] >= "2017-07-31"].copy()
model = lgb.Booster(model_file='../models/lgbm_holidays.txt')
FEATURES = [c for c in df.columns if c not in ["date", "unit_sales"]]

print(f"Test set: {len(test):,} rows | {test['date'].min().date()} to {test['date'].max().date()}")
print("Never seen during training, tuning, or feature selection.")

Test set: 124,039 rows | 2017-07-31 to 2017-08-15
Never seen during training, tuning, or feature selection.


In [2]:
# Predict and score

def mae(y,p):  return np.mean(np.abs(y-p))
def rmse(y,p): return np.sqrt(np.mean((y-p)**2))
def wape(y,p): return np.sum(np.abs(y-p))/np.sum(y)*100
def rmsle(y,p):
    p = np.clip(p,0,None)
    return np.sqrt(np.mean((np.log1p(p)-np.log1p(y))**2))

y_test = test["unit_sales"].values

test["pred_lgbm"]     = np.clip(np.expm1(model.predict(test[FEATURES])), 0, None)
test["pred_optimal"]  = test["pred_lgbm"] * 0.80          # factor chosen on validation
test["pred_baseline"] = test["roll_mean_7"]

rows = []
for name, col in [("Moving avg 7d (baseline)","pred_baseline"),
                  ("LightGBM","pred_lgbm"),
                  ("LightGBM + 0.80 bias","pred_optimal")]:
    p = test[col].values
    rows.append({"model":name, "MAE":round(mae(y_test,p),4), "RMSE":round(rmse(y_test,p),4),
                 "WAPE":round(wape(y_test,p),2), "RMSLE":round(rmsle(y_test,p),4)})

print(pd.DataFrame(rows).to_string(index=False))

                   model    MAE    RMSE      WAPE  RMSLE
Moving avg 7d (baseline) 3.9162 11.4179 48.599998 0.7023
                LightGBM 3.3600  9.1422 41.700000 0.5963
    LightGBM + 0.80 bias 3.7382 10.0224 46.390000 0.6177


In [3]:
# Cost on test 

COST_OVER, COST_UNDER = 6.00, 2.00

def forecast_cost(actual, forecast, co=COST_OVER, cu=COST_UNDER):
    e = forecast - actual
    return np.clip(e,0,None)*co + np.clip(-e,0,None)*cu

costs = {n: forecast_cost(test["unit_sales"], test[c]).sum()
         for n,c in [("Baseline","pred_baseline"),("LightGBM","pred_lgbm"),("LightGBM + bias","pred_optimal")]}

print("\n16-day test period cost:")
for n,v in costs.items():
    print(f"  {n:<18} ${v:>12,.0f}")
print(f"\nReduction (LightGBM):        {(costs['Baseline']-costs['LightGBM'])/costs['Baseline']*100:.1f}%")
print(f"Reduction (LightGBM + bias): {(costs['Baseline']-costs['LightGBM + bias'])/costs['Baseline']*100:.1f}%")


16-day test period cost:
  Baseline           $   1,973,964
  LightGBM           $   1,371,719
  LightGBM + bias    $   1,218,649

Reduction (LightGBM):        30.5%
Reduction (LightGBM + bias): 38.3%


In [4]:
test.to_parquet('../data/processed/test_with_preds.parquet', index=False)
pd.DataFrame(rows).to_csv('../reports/final_test_results.csv', index=False)
print("Saved.")

Saved.
